In [2]:
import json
import os
from time import sleep
from urllib import parse
import requests
import urllib.request
import string
import re
import pandas as pd
from datetime import date, datetime,timedelta
import urllib.parse
from sqlalchemy import text
import sqlalchemy
import random
import shutil
import sys
import os

# 获取模块的绝对路径
module_path = r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\弘则"

# 检查模块路径是否已经在 sys.path 中，避免重复添加
if module_path not in sys.path:
    # 将模块路径添加到 sys.path 开头，确保优先加载这个路径的模块
    sys.path.insert(0, module_path)

import timai
from timai import get_html_text_and_images,client_hz,summarize_content,automaton,filter_sensitive_words

folder_path=r'D:\2024年\企业预警通\新建未找到文件夹\新建未找到文件夹\新建未找到文件夹'
sql_engine = sqlalchemy.create_engine(
  'mysql+pymysql://%s:%s@%s:%s/%s' % (
    'hz_work',
    'Hzinsights2015',
    'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
    '3306',
    'yq',
  ), poolclass=sqlalchemy.pool.NullPool
  )

sql="""
SELECT A.trade_code,A.fileName,
A.公司名称 as company
from yq.23年财报文件 A
where A.fileName !=''
"""

with sql_engine.begin() as connection:
    df = pd.read_sql(sql, connection)

def sanitize_filename(filename):
    """清理文件名，使其符合Windows文件命名规范"""
    # 替换不允许的字符为下划线
    sanitized = re.sub(r'[<>:"/\\|?*]', '_', filename)
    # 注意：此处未处理可能存在的结束点字符（.）连续情况，根据需要进一步处理
    return sanitized
def ocr_database(originfilename,beizhu):
  sql = """
  UPDATE 23年财报文件
  set ocr= :beizhu
  where fileName = :originfilename
  """
  # 构建参数字典
  params = {
    "originfilename":originfilename,
    "beizhu":beizhu
  }
  with sql_engine.begin() as connection:
    connection.execute(text(sql), params)

def download_PDF(fileUrl,fileName):  #下载pdf
  file_path = os.path.join(folder_path,fileName+".pdf")
  # 检查文件是否已存在，如果存在则不下载
  if os.path.exists(file_path):
    print(f"{fileName}File already exists!")
  else:
    r = requests.get(fileUrl)
    with open(file_path, "wb") as f:
      f.write(r.content)
    print(f"Downloaded: {fileName}.pdf\n")

def update_database(list_keywords,TypeInfo,happenDate,ContentInfo,ai_sum):
  sql = """
  INSERT ignore INTO 金融债舆情监控 (list_keywords,TypeInfo,happenDate,ContentInfo,ai_sum)
  VALUES (:list_keywords, :TypeInfo, :happenDate, :ContentInfo, :ai_sum)
  """
  # 构建参数字典
  params = {
    "list_keywords": list_keywords,
    "TypeInfo": TypeInfo,
    "happenDate":happenDate,
    "ContentInfo": ContentInfo,
    "ai_sum": ai_sum
  }
  with sql_engine.begin() as connection:
    connection.execute(text(sql), params)
def update_database1(trade_code,fileUrl,fileName):
  sql = """
  UPDATE 23年财报文件
  set fileUrl= :fileUrl, fileName= :fileName, ocr='已重下'
  where trade_code = :trade_code
  """
  # 构建参数字典
  params = {
    "trade_code": trade_code,
    "fileUrl": fileUrl,
    "fileName":fileName
  }
  with sql_engine.begin() as connection:
    connection.execute(text(sql), params)

headers = {
   "Content-Type": "application/json", 
   "Authorization": "token 2b7ky64rjarrrcz1"
 }


def pro_id(id,headers=headers_yy):
  url=f"https://host.ratingdog.cn/api/services/app/ResearchInfo/GetCiPublicOpinionForView?id={id}"
  data={
    "id":id 
  }
  r = requests.get(url, headers=headers, json=data)
  r = str(r.content, encoding="utf-8")
  r = json.loads(r)
  #提取关键公司
  if r['result'] is None:
     return None
  keywords=r['result']['miniIssuers']
  list_keywords=''
  for i in range(len(keywords)):
      list_keywords+=keywords[i]['issuerName']+','

  #发生日期
  happenDate=r['result']['happenDate']
  #信息类型
  TypeInfo_part1 = r['result'].get('parentPublicOpinionTypeName')
  TypeInfo_part2 = r['result'].get('publicOpinionTypeName')

  TypeInfo = ''
  if TypeInfo_part1 is not None:
      TypeInfo += TypeInfo_part1
  if TypeInfo_part2 is not None:
      TypeInfo += TypeInfo_part2


  #信息内容
  content, image_texts=get_html_text_and_images(html_path=0, html_content=r['result']['publicOpinionContent'])
  commentary=r['result']['commentary']
  if r['result']['commentary']==None:
      commentary=''
  if content==None:
    content=''
  if image_texts==None:
      image_texts=''
  ContentInfo=commentary+content+str(image_texts)
  content0='涉及公司：'+list_keywords+'\n'+'负面信息类型：'+TypeInfo+'\n'+'事件发生日期：'+happenDate+'\n'+'具体负面信息内容：'+ContentInfo
  content0=filter_sensitive_words(content0, automaton)
  ai_sum=summarize_content(content=content0,type='舆情',client=client_hz)

  return list_keywords,TypeInfo,happenDate,ContentInfo,ai_sum


def pro_yyinfo(dt_start,dt_end,headers=headers_yy):
  dt_start = datetime.strptime(dt_start, "%Y-%m-%d")
  dt_end = datetime.strptime(dt_end, "%Y-%m-%d")
  SkipCount=0
  dt=dt_start
  while dt<=dt_end:
    dt_str=dt.strftime("%Y-%m-%d")
    dt+=timedelta(days=1)
    print(dt_str)
    is_end=0
    while is_end==0:
      data={
        "IsRisk": True,
        "MoodLevels": [],
        "StartDate": dt_str,
        "EndDate": dt_str,
        "Type": "",
        "administrativeDivisionIds": [],
        "industryIds": [
          "39ff4c48-a257-0b76-71f7-13fee3262f49",
          "39ff4c48-a260-4c75-895a-5f35fca09011",
          "39ff4c48-a265-b80e-5c68-87eaf63dc085",
          "39ff4c48-a268-1697-6aa8-fb097d58c81e",
          "39ff4c48-a26b-d5c4-49fd-6640cb7ff50e",
          "39ff4c48-a26e-855a-32ab-1625a06ea28b",
          "39ff4c48-a271-2834-7830-bdce9b67538e",
          "39ff4c48-a275-5b66-5db3-1f01ad36c93e",
          "3a0d0503-23f7-c800-1479-d9612aaf8495",
          "39ff4c48-a2ad-a3f0-bce5-39bd7b10d006",
          "39ff4c48-a2b1-6c7b-34bf-68e4304424a0",
          "39ff4c48-a348-0cf0-aaee-864c8ea41414",
          "39ff4c48-a34b-2b1d-030e-8af60e1da6e3",
          "39ff4c48-a34e-40a2-0452-06be2d97dc49",
          "39ff4c48-a351-5810-1e2e-8197ca5744c6",
          "39ff4c48-a353-1f9d-7fe2-e3974c9c9e0e",
          "39ff4c48-a356-9ce2-2f02-53f4e57fc514",
          "39ff4c48-a359-217b-bfd0-5128f2467f99",
          "39ff4c48-a35c-c6d1-0715-8c6966db9483",
          "39ff4c48-a35f-e100-e27e-101edd28245f",
          "39ffb434-fcd6-5edf-b456-0c656e7e5cf7",
          "3a12922c-fa86-fef5-5ad5-da6c7fa5f74a",
          "3a12924d-a88d-e831-e748-dfc3388e958a",
          "3a129257-9559-a547-37e1-fb1fadd09bb4"],
        "Source": False,
        "MaxResultCount": 30,
        "SkipCount": SkipCount
      }

      url = "https://host.ratingdog.cn/api/services/app/ResearchInfo/GetQueryPublicOpinionsForTenant"
      requests.DEFAULT_RETRIES = 5
      s = requests.session()    
      r = requests.post(url, headers=headers, json=data)
      r = str(r.content, encoding="utf-8")
      r = json.loads(r)
      print(r)
      totalCount=r['result']['totalCount']
      if totalCount>0:
        for item in r['result']['items']:
          results = pro_id(item['id'])
          if results is None:
             continue
          results = [r or '' for r in results]
          list_keywords, TypeInfo, happenDate, ContentInfo, ai_sum = results
          print(ai_sum)
          update_database(list_keywords,TypeInfo,happenDate,ContentInfo,ai_sum)
      else:
        is_end=1
      if SkipCount+30>totalCount:
        is_end=1
      else:
        SkipCount+=30

         
# pro_yyinfo('2022-09-30','2023-12-31')

NameError: name 'headers_yy' is not defined

In [13]:
url_lsnote='http://127.0.0.1:6806/api/notebook/lsNotebooks'
url_insnote='http://127.0.0.1:6806/api/filetree/createDocWithMd'
data={
  "notebook": "20240711131333-a9cy8wy",
  "path": "/daily note",
  "markdown": "#13232"
}
res=requests.post(url_insnote,headers=headers,json=data)
res
# r = str(res.content, encoding="utf-8")
# r


<Response [200]>

In [16]:
df=pd.read_excel(r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\弘则\银行数据库.xlsx")
with sql_engine.begin() as connection:
    df.to_sql('银行数据库',connection,index=False)

In [9]:
content0='''涉及公司：江苏盐城农村商业银行股份有限公司,
负面信息类型：公司治理高管违法违纪
事件发生日期：2024-06-21
具体负面信息内容：公开信息看，李一平自2016.11-2023.7担任公司董事长，任职时间长，尚不清楚其被查原因，持续关注对公司的后续影响。---IMAGE 1---', '江苏盐城农村商业银行原董事长李一平被查\n知道Newis\n据江苏省纪委监委6月20日消息，江苏盐城农村商业银行股份有限公司原党委书记、董\n事长季一平涉嫌严重违纪违法，目前正接受江苏省纪委监委派驻省农村信用社联合社纪\n检监察组纪律审查和盐城市监委监察调查\n1√111\n李一平，图源：新华日很\n据新华日报消息，公开资料显示，李一平，男，1965年生，盐城人，本科学历，中共党\n员，曾任江苏盐城农村商业银行党委书记、董事长，盐城市慈善总会副会长等职，\n'), ('---IMAGE 2---', '李一平简历\n公开信息显示，李一平出生于1965年10月，江苏大丰人，中共党员，大学文化\n经济师\n李一平长期在江苏盐城农信系统工作，早年曾在大丰联社及多个当地信用社任职。\n2003年8月，李一平任大丰联社主任。\n2005年11月任大丰农村合作银行行长。\n2009年6月至2012年5月，李一平任滨海农村信用合作联社理事长。\n2012年6月至2014年1月，任滨海农商银行董事长。\n2014年1月至2016年10月，任建湖农商银行董事长，\n2016年11月至2023年6月，李一平任盐城农商银行董事长。\n2023年7月，盐城农商银行召开2023年第一次临时股东大会，选举产生新一届董事\n会成员，李一平未在董事名单中\n大眼查信息显示，2024年1月，盐城农商银行法定代表人由季一平变更为他人。\n'''
from timai import automaton,filter_sensitive_words
content0=filter_sensitive_words(content0, automaton)
summarize_content(content=content0,type='舆情',client=client_hz)


'结论：董事长涉嫌违法对盐城农商银行的信用及偿付能力构成中度风险，短期内可能影响市场信心，长期看需关注接任者表现及经营稳定性。\n\n逻辑：高管违法事件暴露了公司治理问题，可能导致监管处罚、经营动荡，进而影响财务状况。市场可能会据此调整对公司债券的评级，增加融资成本。不过，目前尚未有实际财务影响的证据，需观察事件进展。\n\n关注：1. 监管反应与处罚措施；2. 公司内部管理层调整及经营稳定性；3. 资金流动性变化；4. 债券市场交易价格波动。信息缺口在于事件的具体细节和对公司运营的实际影响，需持续跟踪官方公告及财务报告。'

In [1]:
content0="涉及公司：奇虎360科技有限公司,上海淇毓信息科技有限公司\n负面信息类型：财务风险\n事件发生日期：2024-05-13\n具体负面信息内容：上海淇毓（YY评级7）是奇富科技（3660.HK）的运营主体，其控股股东上海奇步天下是奇富科技的关联方。奇富科技前身是“360金融”，于22年从360集团中分拆出来，主要从事撮合金融机构贷款和自有资金放贷业务，分别占比84.3%和15.7%。在撮合贷款业务中尽管奇富科技不直接发放贷款，但需要为这部分贷款提供担保或反担保，承担信贷风险，截至2023年底，奇富科技的备用担保负债为39.5亿元、或有担保负债为32.07亿元。目前，在经济下行周期，奇富科技撮合贷款的逾期率正在逐年上升，据年报披露，奇富科技21-23年的90天+逾期率分别为1.54%、2.03%及2.35%，资产质量压力持续攀升。[('---IMAGE 1---', '')]"